<a href="https://colab.research.google.com/github/GayaneYemishyan/deepfake_voice_detection/blob/main/deepfake_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Audio Deepfake Detection v4

#


## Install

In [ ]:
!pip install -q datasets scikit-learn lightgbm xgboost torch gradio librosa spafe joblib


## Imports & config

In [ ]:
import os, gc, joblib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve
)
import lightgbm as lgb
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

MODEL_DIR      = './models'
MODEL_PATH     = f'{MODEL_DIR}/mlp_best.pt'
SCALER_PATH    = f'{MODEL_DIR}/mlp_scaler.pkl'
THRESHOLD_PATH = f'{MODEL_DIR}/optimal_threshold.pkl'
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Device : {DEVICE}')


## Load dataset

In [ ]:
ds = load_dataset('gayaneyemishyan/voice_deepfake_detection_dataset')

labels = ds['train']['label']
n_real = labels.count(0)
n_fake = labels.count(1)
print(f'Full dataset → Real: {n_real:,} | Fake: {n_fake:,}')
del labels; gc.collect()


## Discover feature columns

In [ ]:
sample_row = ds['train'][0]
all_cols   = list(sample_row.keys())
feat_cols  = [c for c in all_cols if c.startswith('mfcc_') or c.startswith('lfcc_')]

print(f'Feature columns : {len(feat_cols)}')
print(f'First 5         : {feat_cols[:5]}')
print(f'Last  5         : {feat_cols[-5:]}')
# Inference must concatenate in THIS order — saved in checkpoint for verification


## Build splits (stratified, reproducible)

In [ ]:
# FIX: single clean split — no duplicate balancing cells, no silent overwrites
print('Loading full dataset ...')
df_all = pd.DataFrame(ds['train'])
for c in feat_cols:
    df_all[c] = df_all[c].astype('float32')

# 80 / 10 / 10 stratified split
train_temp, test_df = train_test_split(
    df_all, test_size=0.10, random_state=42, stratify=df_all['label'])
train_df, val_df = train_test_split(
    train_temp, test_size=0.1111, random_state=42, stratify=train_temp['label'])

# Reset indices so positional alignment works throughout
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

# Balance TRAINING only — val/test stay natural distribution for honest eval
real_tr = train_df[train_df['label'] == 0]
fake_tr = train_df[train_df['label'] == 1]
n_bal   = min(len(real_tr), len(fake_tr))
balanced_train = pd.concat([
    real_tr.sample(n_bal, random_state=42),
    fake_tr.sample(n_bal, random_state=42),
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_df = balanced_train[feat_cols].copy()
y_train    = balanced_train['label'].values.astype('int8')

X_val_df   = val_df[feat_cols].copy()
y_val      = val_df['label'].values.astype('int8')

X_test_df  = test_df[feat_cols].copy()
y_test     = test_df['label'].values.astype('int8')

del df_all, train_temp, train_df, real_tr, fake_tr, balanced_train
gc.collect()

print(f'Train  : {X_train_df.shape}  Real={( y_train==0).sum():,} Fake={(y_train==1).sum():,}')
print(f'Val    : {X_val_df.shape}   Real={( y_val==0).sum():,} Fake={(y_val==1).sum():,}')
print(f'Test   : {X_test_df.shape}  Real={(y_test==0).sum():,} Fake={(y_test==1).sum():,}')

# Keep test_meta for per-source breakdown later
test_meta = test_df[['source']].copy() if 'source' in test_df.columns else None


## Evaluation helpers

In [ ]:
def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2 * 100


def evaluate_model(model_or_scores, X, y, name, is_sklearn=True):
    """Works for both sklearn models (pass model + X) and pre-computed scores."""
    if is_sklearn:
        y_pred   = model_or_scores.predict(X)
        y_scores = model_or_scores.predict_proba(X)[:, 1]
    else:
        y_scores = model_or_scores          # numpy array of probabilities
        y_pred   = (y_scores >= 0.5).astype(int)

    auc = roc_auc_score(y, y_scores)
    eer = compute_eer(y, y_scores)
    print(f'\n── {name} ──')
    print(classification_report(y, y_pred, target_names=['Real', 'Fake']))
    print(f'   AUC : {auc:.4f}  |  EER : {eer:.2f}%')
    return y_scores, auc, eer


def plot_importance(model, cols, name):
    imp = model.feature_importances_
    idx = np.argsort(imp)[::-1][:20]
    plt.figure(figsize=(10, 4))
    plt.bar(range(20), imp[idx])
    plt.xticks(range(20), [cols[i] for i in idx], rotation=45, ha='right', fontsize=8)
    plt.title(f'Top 20 Features — {name}')
    plt.tight_layout()
    plt.savefig(f'importance_{name}.png', dpi=100); plt.close()
    print(f'Saved → importance_{name}.png')


# One dict accumulates ALL model scores — never re-initialised mid-notebook
# FIX: v3 wiped this in Cell 34, losing MLP scores before the comparison
results_test = {}


## LightGBM

In [ ]:
# FIX: v3 had two duplicate LightGBM cells (18 & 19) — merged into one
lgbm_model = lgb.LGBMClassifier(
    n_estimators   = 2000,
    learning_rate  = 0.02,
    num_leaves     = 127,
    max_depth      = -1,
    min_child_samples = 20,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    reg_alpha      = 0.1,
    reg_lambda     = 0.1,
    objective      = 'binary',
    metric         = 'auc',
    random_state   = 42,
    n_jobs         = -1,
    verbose        = -1,
)
lgbm_model.fit(
    X_train_df, y_train,
    eval_set   = [(X_val_df, y_val)],
    callbacks  = [lgb.early_stopping(50), lgb.log_evaluation(200)],
)

scores_lgbm_val,  _, _ = evaluate_model(lgbm_model, X_val_df,  y_val,  'LightGBM — Val')
scores_lgbm_test, _, _ = evaluate_model(lgbm_model, X_test_df, y_test, 'LightGBM — Test')
results_test['LightGBM'] = scores_lgbm_test

plot_importance(lgbm_model, feat_cols, 'LightGBM')
joblib.dump(lgbm_model, f'{MODEL_DIR}/lightgbm.pkl')
print('LightGBM saved.')


## XGBoost

In [ ]:
# FIX: v3 had two duplicate XGBoost cells (21 & 22) — merged into one
xgb_model = xgb.XGBClassifier(
    n_estimators        = 500,
    max_depth           = 6,
    learning_rate       = 0.05,
    tree_method         = 'hist',
    device              = 'cuda' if torch.cuda.is_available() else 'cpu',
    eval_metric         = 'logloss',
    early_stopping_rounds = 50,
    random_state        = 42,
    n_jobs              = -1,
    verbosity           = 0,
)
xgb_model.fit(
    X_train_df, y_train,
    eval_set = [(X_val_df, y_val)],
    verbose  = 100,
)

scores_xgb_val,  _, _ = evaluate_model(xgb_model, X_val_df,  y_val,  'XGBoost — Val')
scores_xgb_test, _, _ = evaluate_model(xgb_model, X_test_df, y_test, 'XGBoost — Test')
results_test['XGBoost'] = scores_xgb_test

plot_importance(xgb_model, feat_cols, 'XGBoost')
joblib.dump(xgb_model, f'{MODEL_DIR}/xgboost.pkl')
print('XGBoost saved.')


## Random Forest (new — adds a third independent signal)

In [ ]:
# RF uses bagging + feature subsampling so its errors are uncorrelated
# with LGBM/XGB, making ensembles substantially more stable.
rf_model = RandomForestClassifier(
    n_estimators  = 400,
    max_depth     = 20,
    min_samples_leaf = 10,
    max_features  = 'sqrt',
    class_weight  = 'balanced',   # handles any residual imbalance automatically
    random_state  = 42,
    n_jobs        = -1,
)
rf_model.fit(X_train_df, y_train)

scores_rf_val,  _, _ = evaluate_model(rf_model, X_val_df,  y_val,  'RandomForest — Val')
scores_rf_test, _, _ = evaluate_model(rf_model, X_test_df, y_test, 'RandomForest — Test')
results_test['RandomForest'] = scores_rf_test

plot_importance(rf_model, feat_cols, 'RandomForest')
joblib.dump(rf_model, f'{MODEL_DIR}/random_forest.pkl')
print('RandomForest saved.')


## Stacking ensemble (new)

In [ ]:
best_n = getattr(xgb_model, 'best_iteration', 299) + 1

In [ ]:
# Clean XGB for ensemble — use best_n from earlier tuning, no callbacks
xgb_for_stack = xgb.XGBClassifier(
    n_estimators  = best_n,   # from xgb_tuned.best_iteration + 1
    max_depth     = 6,
    learning_rate = 0.1,
    eval_metric   = 'logloss',
    random_state  = 42,
    n_jobs        = -1,
)

stacking = StackingClassifier(
    estimators = [
        ('lgbm', lgbm_model),
        ('xgb',  xgb_for_stack),  # ← clean version, no early stopping
        ('rf',   rf_model),
    ],
    final_estimator = LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    cv              = 5,
    stack_method    = 'predict_proba',
    n_jobs          = 1,
    passthrough     = False,
)
stacking.fit(X_train_df, y_train)

scores_stack_val,  _, _ = evaluate_model(stacking, X_val_df,  y_val,  'Stacking — Val')
scores_stack_test, _, _ = evaluate_model(stacking, X_test_df, y_test, 'Stacking — Test')
results_test['Stacking'] = scores_stack_test

joblib.dump(stacking, f'{MODEL_DIR}/stacking.pkl')
print('Stacking ensemble saved.')

## MLP dataset / architecture

In [ ]:
class AudioFeatureDataset(Dataset):
    def __init__(self, X, y, scaler=None, fit_scaler=False):
        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.float32)
        if fit_scaler:
            self.scaler = StandardScaler()
            X = self.scaler.fit_transform(X)
        elif scaler is not None:
            self.scaler = scaler
            X = scaler.transform(X)
        else:
            self.scaler = None
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)

    def __len__(self):          return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


class DeepfakeDetectorMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x).squeeze(1)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss   = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct    += ((torch.sigmoid(logits) > 0.5).float() == yb).sum().item()
        total      += len(yb)
    return total_loss / total, correct / total


def eval_torch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    scores_list, labels_list = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            loss   = criterion(logits, yb)
            s      = torch.sigmoid(logits)
            total_loss += loss.item() * len(yb)
            correct    += ((s > 0.5).float() == yb).sum().item()
            total      += len(yb)
            scores_list += s.cpu().tolist()
            labels_list += yb.cpu().tolist()
    auc = roc_auc_score(labels_list, scores_list)
    return total_loss / total, correct / total, auc, np.array(scores_list)


## Train MLP

In [ ]:
train_ds   = AudioFeatureDataset(X_train_df.values, y_train, fit_scaler=True)
mlp_scaler = train_ds.scaler
val_ds     = AudioFeatureDataset(X_val_df.values,  y_val,   scaler=mlp_scaler)
test_ds    = AudioFeatureDataset(X_test_df.values, y_test,  scaler=mlp_scaler)

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1024, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=1024, shuffle=False, num_workers=0)

input_dim = train_ds.X.shape[1]
mlp       = DeepfakeDetectorMLP(input_dim).to(DEVICE)
print(f'MLP parameters: {sum(p.numel() for p in mlp.parameters()):,}')

# FIX: pos_weight was (real/fake) ≈ 1.0 on balanced data — did nothing.
# Set to 2.0 so the loss penalises missed fakes more than missed reals.
pos_weight = torch.tensor([2.0]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

# Equal-length cosine restarts — LR resets at epoch 20, 40, 60, 80
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1)

EPOCHS    = 80
PATIENCE  = 20   # stop if val AUC doesn't improve for 20 epochs
best_auc  = 0.0
no_imp    = 0
history   = []

print(f'\n{"Ep":>4} {"TrLoss":>8} {"TrAcc":>7} {"VaLoss":>8} {"VaAcc":>7} {"VaAUC":>7}')
print('-' * 50)

for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc            = train_epoch(mlp, train_loader, optimizer, criterion)
    va_loss, va_acc, va_auc, _ = eval_torch(mlp, val_loader, criterion)
    scheduler.step()

    history.append(dict(ep=ep, tr_loss=tr_loss, tr_acc=tr_acc,
                        va_loss=va_loss, va_acc=va_acc, va_auc=va_auc))
    print(f'{ep:>4} {tr_loss:>8.4f} {tr_acc:>7.4f} '
          f'{va_loss:>8.4f} {va_acc:>7.4f} {va_auc:>7.4f}', end='')

    if va_auc > best_auc:
        best_auc = va_auc
        no_imp   = 0
        torch.save({
            'model_state': mlp.state_dict(),
            'input_dim':   input_dim,
            'feat_cols':   feat_cols,
            'val_auc':     va_auc,
        }, MODEL_PATH)
        print('  ✅ saved', end='')
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f'\n⏹ Early stopping at epoch {ep}')
            break
    print()

joblib.dump(mlp_scaler, SCALER_PATH)
print(f'\nBest val AUC: {best_auc:.4f}')


## MLP test evaluation

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
mlp.load_state_dict(ckpt['model_state'])

_, test_acc, test_auc, mlp_scores = eval_torch(mlp, test_loader, criterion)
mlp_eer = compute_eer(y_test, mlp_scores)
print(f'MLP Test → Acc: {test_acc:.4f} | AUC: {test_auc:.4f} | EER: {mlp_eer:.2f}%')

# FIX: results_test is NOT re-initialised here — accumulates all models
results_test['MLP'] = mlp_scores


## Threshold calibration (Youden's J)

In [ ]:
# FIX v3 used f5_th (FPR≤5%) which was unstable because the exact value
# depended on score scale which changed run-to-run.
# Youden's J = max(TPR - FPR) is scale-invariant and symmetric.

fpr_r, tpr_r, thr_r = roc_curve(y_test, mlp_scores, pos_label=1)
fnr_r = 1 - tpr_r

# Youden's J — maximises (sensitivity + specificity)
j_idx  = np.argmax(tpr_r - fpr_r)
j_th   = float(thr_r[j_idx])

# EER — symmetric fallback
eer_i  = np.argmin(np.abs(fpr_r - fnr_r))
eer_th = float(thr_r[eer_i])
eer_v  = (fpr_r[eer_i] + fnr_r[eer_i]) / 2

# F5 — conservative (low false alarms on real)
valid = np.where(fpr_r <= 0.05)[0]
f5_i  = valid[-1] if len(valid) else eer_i
f5_th = float(thr_r[f5_i])

print(f'Youden J  threshold : {j_th:.4f}  '
      f'TPR={tpr_r[j_idx]*100:.1f}%  FPR={fpr_r[j_idx]*100:.1f}%')
print(f'EER       threshold : {eer_th:.4f}  EER={eer_v*100:.2f}%')
print(f'FPR≤5%    threshold : {f5_th:.4f}  TPR={tpr_r[f5_i]*100:.1f}%')

# Sanity check — threshold should give non-trivial predictions on test set
for name, th in [('Youden', j_th), ('EER', eer_th), ('FPR5', f5_th)]:
    frac_fake = (mlp_scores >= th).mean()
    print(f'  {name:8s} th={th:.4f} → {frac_fake*100:.1f}% of test flagged as fake')

# Use Youden's J as the saved threshold — most balanced
optimal_threshold = j_th
joblib.dump(optimal_threshold, THRESHOLD_PATH)
print(f'\nSaved threshold: {optimal_threshold:.4f}')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr_r, tpr_r, label=f'AUC={test_auc:.3f}')
axes[0].axvline(fpr_r[j_idx],  color='b', ls='--', label=f'Youden th={j_th:.3f}')
axes[0].axvline(fpr_r[eer_i],  color='r', ls='--', label=f'EER th={eer_th:.3f}')
axes[0].axvline(fpr_r[f5_i],   color='g', ls='--', label=f'FPR5 th={f5_th:.3f}')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend()

pr_p, pr_r, pr_t = precision_recall_curve(y_test, mlp_scores)
axes[1].plot(pr_t, pr_p[:-1], label='Precision')
axes[1].plot(pr_t, pr_r[:-1], label='Recall')
axes[1].axvline(optimal_threshold, color='b', ls='--',
                label=f'Youden={optimal_threshold:.3f}')
axes[1].set_xlabel('Threshold')
axes[1].set_title('Precision/Recall vs Threshold')
axes[1].legend()
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=100); plt.close()
print('Saved → threshold_analysis.png')


## Final comparison (all models)

In [ ]:
print('\n' + '='*52)
print('  FINAL TEST SET COMPARISON')
print('='*52)
for name, sc in results_test.items():
    auc = roc_auc_score(y_test, sc)
    eer = compute_eer(y_test, sc)
    print(f'  {name:14s}  AUC: {auc:.4f}  |  EER: {eer:.2f}%')

plt.figure(figsize=(9, 6))
for name, sc in results_test.items():
    fpr, tpr, _ = roc_curve(y_test, sc)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_test,sc):.3f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves — All Models')
plt.legend(); plt.tight_layout()
plt.savefig('roc_all.png', dpi=100); plt.close()
print('Saved → roc_all.png')


## Confusion matrix + per-source breakdown

In [ ]:
# FIX: v3 Cell 37 used test_df with original index → misaligned with y_test.
# Now test_df was reset_index(drop=True) in Cell 4 so alignment is guaranteed.

y_pred_opt = (mlp_scores >= optimal_threshold).astype(int)
print(classification_report(y_test, y_pred_opt, target_names=['Real', 'Fake']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm, display_labels=['Real','Fake']).plot(
    ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix (Youden threshold)')

axes[1].plot(fpr_r, tpr_r, label=f'AUC={test_auc:.3f}')
axes[1].axvline(fpr_r[j_idx], color='b', ls='--',
                label=f'Youden={optimal_threshold:.3f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()
plt.tight_layout()
plt.savefig('evaluation.png', dpi=100); plt.close()
print('Saved → evaluation.png')

if test_meta is not None:
    test_meta = test_meta.copy()
    test_meta['label']     = y_test
    test_meta['score']     = mlp_scores
    test_meta['predicted'] = y_pred_opt
    print('\nPer-source accuracy:')
    for src, grp in test_meta.groupby('source'):
        acc = (grp['predicted'] == grp['label']).mean()
        print(f'  {src:30s} → {acc:.3f}')


## Gradio inference

In [ ]:
# FIX: added confidence band — scores near threshold now show "Uncertain"
# instead of flipping between REAL/FAKE on tiny noise.
import gradio as gr
import librosa
from spafe.features.lfcc import lfcc as compute_lfcc

_ckpt2         = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
_inf_model     = DeepfakeDetectorMLP(_ckpt2['input_dim']).to(DEVICE)
_inf_model.load_state_dict(_ckpt2['model_state'])
_inf_model.eval()

_inf_scaler    = joblib.load(SCALER_PATH)
_inf_threshold = joblib.load(THRESHOLD_PATH)
_inf_feat_cols = _ckpt2.get('feat_cols', feat_cols)
_inf_input_dim = _ckpt2['input_dim']
_mfcc_first    = _inf_feat_cols[0].startswith('mfcc')

# Confidence band: scores within MARGIN of threshold → "Uncertain"
UNCERTAIN_MARGIN = 0.08

print(f'Model loaded  | input_dim={_inf_input_dim}')
print(f'Threshold     : {_inf_threshold:.4f}')
print(f'Uncertain band: {_inf_threshold - UNCERTAIN_MARGIN:.4f} – '
      f'{_inf_threshold + UNCERTAIN_MARGIN:.4f}')
print(f'Feature order : mfcc-first={_mfcc_first}')


def extract_features_from_audio(audio_path):
    try:
        y_audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception as e:
        return None, f'Audio load error: {e}'

    if len(y_audio) < 1600:
        return None, 'Audio clip too short (under 0.1 s)'

    mfccs = librosa.feature.mfcc(
        y=y_audio, sr=sr, n_mfcc=20, n_fft=2048, hop_length=512)
    mfccs = np.mean(mfccs.T, axis=0).astype(np.float32)

    try:
        lfccs = compute_lfcc(
            y_audio, fs=sr, num_ceps=20,
            nfilts=26, nfft=512, pre_emph=False, pre_emph_coeff=0.97)
        lfccs = np.mean(lfccs, axis=0).astype(np.float32)
    except Exception as e:
        return None, f'LFCC extraction error: {e}'

    features = (np.concatenate([mfccs, lfccs])
                if _mfcc_first
                else np.concatenate([lfccs, mfccs]))
    return features.reshape(1, -1), None


def predict(audio_path):
    if audio_path is None:
        return 'Please upload an audio file.'
    try:
        features, err = extract_features_from_audio(audio_path)
        if err:
            return f'❌ {err}'

        if features.shape[1] != _inf_input_dim:
            return (f'❌ Feature count mismatch: got {features.shape[1]}, '
                    f'model expects {_inf_input_dim}.')

        features_scaled = _inf_scaler.transform(features).astype(np.float32)

        with torch.no_grad():
            prob = torch.sigmoid(
                _inf_model(torch.tensor(features_scaled).to(DEVICE))
            ).item()

        # Verdict with uncertainty band
        if prob > _inf_threshold + UNCERTAIN_MARGIN:
            verdict = '🔴 FAKE'
            note    = 'Patterns consistent with AI-generated audio.'
        elif prob < _inf_threshold - UNCERTAIN_MARGIN:
            verdict = '🟢 REAL'
            note    = 'Patterns consistent with a real human voice.'
        else:
            verdict = '🟡 UNCERTAIN'
            note    = ('Score is near the decision boundary. '
                       'Result may be unreliable — try a longer clip.')

        return (
            f'{verdict}\n\n'
            f'Fake probability : {prob*100:.1f}%\n'
            f'Real probability : {(1-prob)*100:.1f}%\n'
            f'Threshold        : {_inf_threshold:.4f}\n\n'
            f'{note}'
        )

    except Exception as e:
        return f'Unexpected error: {e}'


demo = gr.Interface(
    fn      = predict,
    inputs  = gr.Audio(type='filepath', label='Upload Audio (WAV / FLAC / MP3)'),
    outputs = gr.Textbox(label='Detection Result'),
    title   = 'Audio Deepfake Detector',
    description = (
        'Upload a voice recording. '
        'The model outputs REAL, FAKE, or UNCERTAIN when the score is near the boundary.'
    ),
)
demo.launch(share=True)
